# Lab 5: Linear SVM (3-class)

This notebook is designed to be reusable.

- Default dataset loads from `data/iris_2d.csv`.
- To use your own dataset, set `DATA_PATH` to a CSV with 2 feature columns and a `target` column.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# ---- Config (edit these) ----
DATA_PATH = Path("data/iris_2d.csv")
FEATURE_COLS = ["sepal_length", "petal_length"]
TARGET_COL = "target"
TEST_SIZE = 0.25
RANDOM_STATE = 42

# ---- Load dataset ----
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    required = set(FEATURE_COLS + [TARGET_COL])
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {DATA_PATH}: {sorted(missing)}")
    X = df[FEATURE_COLS].to_numpy(dtype=float)
    y = df[TARGET_COL].to_numpy()
else:
    iris = datasets.load_iris()
    X = iris.data[:, [0, 2]]  # sepal length + petal length
    y = iris.target
    print(f"Note: {DATA_PATH} not found; using sklearn iris dataset instead.")

print("Shape of feature data (X):", X.shape)
print("Shape of target labels (y):", y.shape)

print("First 5 rows of feature data (X):")
print(X[:5])
print()
print("First 5 rows of target labels (y):")
print(y[:5])

# Visualize the dataset
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="winter", edgecolors="k", marker="o")
plt.title("Dataset (Feature 1 vs Feature 2)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Create and train an SVM classifier with a linear kernel
svm_classifier = SVC(kernel="linear", C=1)
svm_classifier.fit(X_train, y_train)

# Predict on the test data
y_pred = svm_classifier.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Plotting the decision boundary (for 2D features)
h = 0.02
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = svm_classifier.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.8)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, marker="o", edgecolors="k", label="Train")
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, marker="^", edgecolors="k", label="Test")
plt.title("SVM Decision Boundary")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.show()
